In [1]:
import glob
import math
import numpy as np
import ROOT
from utils.hist_utils import get_highest_channels, fit_and_scale_mc

ROOT.EnableImplicitMT()
ROOT.gStyle.SetPalette(ROOT.kRainbow)

ROOT.gSystem.Load("/workspace/KLOE/Include/Codes/libLibRec.so")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/Include/Codes/")

ROOT.gInterpreter.Declare('#include "const.h"')

ROOT.KLOE.setGlobalStyle()

channName = {int(k): str(v) for k, v in ROOT.KLOE.channName}
channTitle = {str(k): str(v) for k, v in ROOT.KLOE.channTitle}
channColor = {str(k): int(v) for k, v in ROOT.KLOE.channColor}

In [2]:
# Pobierz wszystkie pliki pasujące do wzorca
all_files = glob.glob(
    "/workspace/root_files/kitt/ALL_PHYS_THREE_PI0_NoSmearing/*.root"
)

all_files += glob.glob("/workspace/root_files/kitt/ALL_PHYS2_THREE_PI0_NoSmearing/*.root")
all_files += glob.glob("/workspace/root_files/kitt/ALL_PHYS3_THREE_PI0_NoSmearing/*.root")
all_files += glob.glob("/workspace/root_files/hal/DATA_THREE_PI0_NoSmearing/*.root")

def get_valid_files(file_list):
    valid = []
    for f in file_list:
        try:
            tfile = ROOT.TFile.Open(f, "READ")
            if tfile and not tfile.IsZombie() and tfile.GetNkeys() > 0:
                valid.append(f)
            else:
                print(f"Pomijam uszkodzony/pusty plik: {f}")
            if tfile:
                tfile.Close()
        except Exception:
            print(f"Błąd otwarcia pliku (pomijam): {f}")
    return valid

all_files = get_valid_files(all_files)

# Scaling factor
lumi_data = 239811.5318
lumi_mc = 351638.6865
scaling_factor = lumi_data / lumi_mc
print(f"Lumi factor: {scaling_factor:.4f}")

# Przekaż czystą listę plików bezpośrednio do RDataFrame
df = ROOT.RDataFrame("h1", all_files)
df_3pi0 = df.Filter("mcflag == 1 && (mctruth == 4 || mctruth == 0 || mctruth == -1)")
df_data = (df
    .Define("Kchrec_5", "Kchrec[5]")
    .Define("Knerec_5", "Knerec[5]")
    .Define("KnerecSix_5", "KnerecSix[5]")
    .Define("MassPi01", "pi01Fit[5]")
    .Define("MassPi02", "pi02Fit[5]")
    .Define("rho_ch", "hypot(Kchrec[6] - ip[0], Kchrec[7] - ip[1])")
    .Define("rho_ne", "hypot(Knerec[6] - ip[0], Knerec[7] - ip[1])")
    .Define("z_ch", "Kchrec[8] - ip[2]")
    .Define("rho_fid_omega", "hypot(Kchrec[6] - Knerec[6], Kchrec[7] - Knerec[7])")
    .Define("z_fid_omega", "Kchrec[8] - Knerec[8]")
    .Filter("mcflag == 0 && mctruth != -1"))
df = (df
    .Define("Kchrec_5", "Kchrec[5]")
    .Define("Knerec_5", "Knerec[5]")
    .Define("KnerecSix_5", "KnerecSix[5]")
    .Define("MassPi01", "pi01Fit[5]")
    .Define("MassPi02", "pi02Fit[5]")
    .Define("rho_ch", "hypot(Kchrec[6] - ip[0], Kchrec[7] - ip[1])")
    .Define("rho_ne", "hypot(Knerec[6] - ip[0], Knerec[7] - ip[1])")
    .Define("z_ch", "Kchrec[8] - ip[2]")
    .Define("rho_fid_omega", "hypot(Kchrec[6] - Knerec[6], Kchrec[7] - Knerec[7])")
    .Define("z_fid_omega", "Kchrec[8] - Knerec[8]")
    .Filter("mctruth != -1 && mctruth != 0 "))

Błąd otwarcia pliku (pomijam): /workspace/root_files/kitt/ALL_PHYS_THREE_PI0_NoSmearing/mk0_initial_analysis_all_phys_THREE_PI0_NoSmearing_3pi0_NONE_205.root
Lumi factor: 0.6820


Warning in <TFile::Init>: file /workspace/root_files/kitt/ALL_PHYS_THREE_PI0_NoSmearing/mk0_initial_analysis_all_phys_THREE_PI0_NoSmearing_3pi0_NONE_205.root probably not closed, trying to recover
Warning in <TFile::Init>: no keys recovered, file has been made a Zombie


## Runs in the files

In [3]:
ROOT.gStyle.SetOptStat(0)
canva = ROOT.TCanvas("canva", "", 800, 800)

hist = df_data.Histo1D(("nrun_data", ";Run Number;Counts", 100, 30300, 41902), "nrun")
hist_mc = df.Histo1D(("nrun_mc", ";Run Number;Counts", 100, 30300, 41902), "nrun")
hist_mc.Scale(scaling_factor)

leg = ROOT.TLegend(0.45, 0.65, 0.88, 0.88)
leg.SetBorderSize(0)
leg.SetFillStyle(0)

hist.SetLineColor(channColor.get("Data", ROOT.kBlack))
hist.SetMarkerStyle(20)
hist_mc.SetLineColor(channColor.get("MC sum", ROOT.kOrange))

leg.AddEntry(hist.GetPtr(), "Data", "pe1")
leg.AddEntry(hist_mc.GetPtr(), "MC Sum", "l")

hist.Draw("PE1")
hist_mc.Draw("HIST SAME")
leg.Draw("SAME")

canva.Draw()

In [4]:
counts = []

counts.append(df_3pi0.Count())
counts.append(df_3pi0.Filter("mctruth == 4 || mctruth == 0").Count())
counts.append(df_3pi0.Filter("mctruth == 4").Count())

errors = [300, 301, 303, 307, 308]
counts_errors = []
combined_error = ""

for error in errors:
    combined_error += f" && errorcode != {error}" if len(combined_error) > 0 else f"errorcode != {error}"
    counts_errors.append(df_3pi0.Filter(combined_error).Count())

all_counts = counts + counts_errors

ROOT.RDF.RunGraphs(all_counts)

for i, val in enumerate(counts):
    counts[i] = val.GetValue()

error_eff = []
rel_error_eff = []

for i, val in enumerate(counts_errors):
    counts_errors[i] = val.GetValue()
    eff = counts_errors[i] / counts[0] if counts[0] > 0 else 1.0
    rel_eff = counts_errors[i] / counts_errors[i-1] if i > 0 else eff
    error_eff.append(eff)
    rel_error_eff.append(rel_eff)

efficiencies = [0, 0, 0]
efficiencies[0] = counts[1] / counts[0] if counts[0] > 0 else 1.0
efficiencies[1] = counts[2] / counts[1] if counts[1] > 0 else 1.0
efficiencies[2] = counts[2] / counts[0] if counts[0] > 0 else 1.0

print(f"Counts total: {counts[0]},\nCounts preselection: {counts[1]},\nCounts selection: {counts[2]}")
print(f"Efficiency total: {efficiencies[2]*100:.2f}%,\nEfficiency preselection: {efficiencies[0]*100:.2f}%,\nEfficiency selection: {efficiencies[1]*100:.2f}%")

for i, eff in enumerate(error_eff):
    print(f"Error {errors[i]}: {eff*100:.2f}% ({rel_error_eff[i]*100:.2f}%) ({counts_errors[i]})")

Counts total: 25292412,
Counts preselection: 18321552,
Counts selection: 18321552
Efficiency total: 72.44%,
Efficiency preselection: 72.44%,
Efficiency selection: 100.00%
Error 300: 90.53% (90.53%) (22897171)
Error 301: 75.22% (83.09%) (19025415)
Error 303: 74.31% (98.79%) (18794286)
Error 307: 72.44% (97.48%) (18321552)
Error 308: 72.44% (100.00%) (18321552)


In [5]:
histos = {}

for mctruth, chann in channName.items():
    if mctruth > 0 and mctruth < 8:
        hist_list = []
        hist_list.append((df.Filter(f"mctruth == {mctruth}")
                           .Histo1D(("h_rho_ch", ";#rho_{ch} [cm];Counts", 201, 0, 12), "rho_ch")))
        hist_list.append((df.Filter(f"mctruth == {mctruth}")
                           .Histo1D(("h_z_ch", ";z_{ch} - z_{IP} [cm];Counts", 201, -6, 6), "z_ch")))

        histos[chann] = hist_list

handles = [h for hist_list in histos.values() for h in hist_list]
        
ROOT.RDF.RunGraphs(handles)

1

In [6]:
highest_chann = get_highest_channels(histos)

In [7]:
ROOT.gStyle.SetOptStat(0)

canvadivision = ROOT.TCanvas("canva", "", 800, 400)
canvadivision.Divide(2, 1)

num_pads = len(next(iter(histos.values())))
legends = []  # Zapobiega usunięciu legend przez Python GC

for i in range(num_pads):
    canvadivision.cd(i + 1)
    
    # Tworzymy legendę na padzie (x1, y1, x2, y2)
    leg = ROOT.TLegend(0.65, 0.65, 0.88, 0.88)
    leg.SetBorderSize(0)
    leg.SetFillStyle(0)  # Przezroczyste tło
    leg.SetTextSize(0.035)
    
    # Sortujemy tak, aby najwyższy kanał dla danego pada był na samym początku
    sorted_channels = sorted(
        histos.items(),
        key=lambda item: item[0] == highest_chann[i],
        reverse=True
    )

    for idx, (chann, hist_list) in enumerate(sorted_channels):
        hist = hist_list[i]
        h = hist.GetValue() if hasattr(hist, "GetValue") else hist
        
        draw_option = "HIST" if idx == 0 else "SAME HIST"

        color = channColor.get(chann, ROOT.kBlack)
        h.SetLineColor(color)
        h.SetLineWidth(2)
        h.Draw(draw_option)
        
        # Wypełnianie legendy (pobieramy ładny tytuł z channTitle, a jeśli brakuje — używamy chann)
        label = channTitle.get(chann, chann)
        leg.AddEntry(h, str(label), "l")

    # Rysujemy legendę na padzie i zapisujemy referencję
    leg.Draw()
    legends.append(leg)

canvadivision.Draw()

Warning in <TCanvas::Constructor>: Deleting canvas with same name: canva


## Optimization of the cut for the control sample tag $K\to\pi^+\pi^-$

In [8]:
df_CS_tag = {}

df_CS_tag["PM"] = df.Filter("hasPM && mctruth != 0 && mctruth != -1")
df_CS_tag["nonPM"] = df.Filter("!hasPM && mctruth != 0 && mctruth != -1")
df_CS_tag["data"] = df_data.Filter("mctruth != -1")

total_counts = {}

total_counts["PM"] = df_CS_tag["PM"].Count()
total_counts["nonPM"] = df_CS_tag["nonPM"].Count()
total_counts["data"] = df_CS_tag["data"].Count()

ROOT.RDF.RunGraphs(list(total_counts.values()))

total_counts = {key: int(val.GetValue() * scaling_factor) if key != 'data' else int(val.GetValue()) for key, val in total_counts.items()}

### Create the histograms

In [9]:
cut_cond = "true"
optimized_variable = "Kchrec_5"
var_title = "m^{inv}_{+-} [MeV/c^{2}]"

In [10]:
basic_hist = {}
modes = {"full": [201, 280, 600], "zoom": [201, 490, 505]}

for mode, hist_config in modes.items():
    hist_list = []

    for data_type in ["PM", "nonPM", "data"]:
        h_title = f"h_{optimized_variable}_{data_type}_{mode}"
        hist_title_list = f";{var_title};Counts"

        hist_list.append(df_CS_tag[data_type].Histo1D((h_title, hist_title_list, *hist_config), optimized_variable))
    
    basic_hist[mode] = hist_list

all_ptrs = [hist for hist_list in basic_hist.values() for hist in hist_list]
ROOT.RDF.RunGraphs(all_ptrs)

# ---

# 3. Modyfikacje histogramów PO wykonaniu RunGraphs (Skalowanie i Dodawanie)
for mode, hist_list in basic_hist.items():
    # Index 0: PM, Index 1: nonPM, Index 2: data
    
    # Skalowanie histogramów MC (PM i nonPM)
    hist_list[0].Scale(scaling_factor)
    hist_list[1].Scale(scaling_factor)

    # Tworzenie sumy MC (PM + nonPM) i dodanie jej jako 4. element (indeks 3)
    h_added = hist_list[0].Clone(f"h_{optimized_variable}_added_{mode}")
    h_added.Add(hist_list[1].GetPtr(), 1.0)
    
    hist_list.append(h_added)

In [11]:
mode = "zoom"

h_calc = basic_hist[mode][0]
h_calc_obj = h_calc.GetValue() if hasattr(h_calc, "GetValue") else h_calc
mean = h_calc_obj.GetMean()
max_bin = h_calc_obj.GetMaximumBin()
max_x = h_calc_obj.GetBinCenter(max_bin)
max_y = h_calc_obj.GetBinContent(max_bin)
rms = h_calc_obj.GetRMS()
# ----------------------------------------------

# Drawing full and zoomed stacked histograms
canva = ROOT.TCanvas("canva", "", 2000, 1000)
canva.Divide(2, 1)

colors = [ROOT.kRed, ROOT.kBlue, ROOT.kBlack, ROOT.kOrange]
titles = [
    "Events with primary K#rightarrow#pi^{+}#pi^{-}", 
    "Events without primary K#rightarrow#pi^{+}#pi^{-}", 
    "Data", 
    "MC sum"
]

stacks = []
legends = []
lines = []

for pad_idx, (mode_name, hist_list) in enumerate(basic_hist.items()):
    canva.cd(pad_idx + 1)
    
    hs = ROOT.THStack(f"hs_{mode_name}", f"{mode_name.capitalize()} mode;{var_title};Counts")
    leg = ROOT.TLegend(0.45, 0.65, 0.88, 0.88)
    leg.SetBorderSize(0)
    leg.SetFillStyle(0)
    
    data_hist = None

    for h_idx, hist in enumerate(hist_list):
        h = hist.GetValue() if hasattr(hist, "GetValue") else hist
        color = colors[h_idx % len(colors)]
        title = titles[h_idx % len(titles)]
        
        # Rozpoznajemy kanał danych po tytule lub indeksie (tutaj np. indeks 2 = "Data")
        if title == "Data":
            # Styl dla punktów eksperymentalnych (PE1)
            h.SetMarkerStyle(20)       # Pełne kropki
            h.SetMarkerSize(0.8)       # Rozmiar punktów
            h.SetMarkerColor(color)
            h.SetLineColor(color)
            h.SetLineWidth(2)
            
            data_hist = h
            leg.AddEntry(h, title, "pe") # "pe" -> punkty z błędami w legendzie
        else:
            # Styl dla MC (histogramy liniowe)
            h.SetLineColor(color)
            h.SetLineWidth(2)
            
            hs.Add(h)
            leg.AddEntry(h, title, "l")  # "l" -> linia w legendzie

    # line_left = ROOT.TLine(497.611 - 2.0, 0.0, 497.611 - 2.0, hs.GetMaximum() * 0.5)
    line_right = ROOT.TLine(4.0, 0.0, 4.0, hs.GetMaximum() * 0.5)

    line_right.SetLineStyle(ROOT.kDashed)
    
    # 1. Rysujemy stack z MC
    hs.Draw("nostack hist")
    line_right.Draw("SAME")
    
    # 2. Rysujemy Dane osobną opcją PE1 na tej samej skali
    if data_hist:
        data_hist.Draw("E1 P SAME")
        
        # Zabezpieczenie skali Y: jeśli Dane są wyższe niż MC, dopasowujemy stack
        max_y_val = max(hs.GetMaximum(), data_hist.GetMaximum()) * 0.6
        hs.SetMaximum(max_y_val)

    leg.Draw()

    stacks.append(hs)
    legends.append(leg)
    lines.append(line_right)

canva._stacks = stacks
canva._legends = legends

canva.Draw()

Warning in <TCanvas::Constructor>: Deleting canvas with same name: canva


### Optimization loop

In [12]:
tag_condition = "rho_ch < 4.0 && abs(z_ch) < 2.0"

df_CS_tag["PM"] = df_CS_tag["PM"].Filter(tag_condition)
df_CS_tag["nonPM"] = df_CS_tag["nonPM"].Filter(tag_condition)
df_CS_tag["data"] = df_CS_tag["data"].Filter(tag_condition)

total_counts_fv = {}

total_counts_fv["PM"] = df_CS_tag["PM"].Count()
total_counts_fv["nonPM"] = df_CS_tag["nonPM"].Count()
total_counts_fv["data"] = df_CS_tag["data"].Count()

ROOT.RDF.RunGraphs(list(total_counts_fv.values()))

total_counts_fv = {key: int(val.GetValue() * scaling_factor) if key != 'data' else int(val.GetValue()) for key, val in total_counts_fv.items()}

In [13]:
cut_values = [rms * float(val) for val in np.arange(0, 5.0, 0.1)]

pm_counts = []
nonpm_counts = []

for val in cut_values:
    condition = f"abs({optimized_variable} - 497.611) < {val}"

    pm_counts.append(df_CS_tag["PM"].Filter(condition).Count())
    nonpm_counts.append(df_CS_tag["nonPM"].Filter(condition).Count())

In [14]:
all_counts = pm_counts + nonpm_counts
ROOT.RDF.RunGraphs(all_counts)

1

In [15]:
pm_counts = [int(val.GetValue() * scaling_factor) for val in pm_counts]
nonpm_counts = [int(val.GetValue() * scaling_factor) for val in nonpm_counts]

In [16]:
purity = []
significance = []
efficiency = {"PM": [], "nonPM": []}

for nonpm, pm in zip(nonpm_counts, pm_counts):
    purity.append(pm / (nonpm + pm) if (nonpm + pm) > 0 else 0.99)
    significance.append(pm / np.sqrt(nonpm + pm) if (nonpm + pm) > 0 else 0.0)
    efficiency["PM"].append(pm / total_counts["PM"]) 
    efficiency["nonPM"].append(nonpm / total_counts["nonPM"]) 

In [17]:
# canva.SetRightMargin(0.3)
canva = ROOT.TCanvas("canva", "", 800, 800)

x_arr = np.array(cut_values, dtype=float)
y_arr = np.array(purity, dtype=float)

effpm_arr = np.array(efficiency["PM"], dtype=float)
effnonpm_arr = np.array(efficiency["nonPM"], dtype=float)

graph = ROOT.TGraph(len(x_arr), x_arr, y_arr)
graph_pm = ROOT.TGraph(len(x_arr), x_arr, effpm_arr)
graph_nonpm = ROOT.TGraph(len(x_arr), x_arr, effnonpm_arr)
graph.SetMaximum(1.2)
graph.SetMinimum(0.0)

graph.SetTitle("")
graph.GetXaxis().SetTitle("Cut value m^{inv}_{+-} [MeV/c^{2}]")
graph.GetYaxis().SetTitle("Value [-]")
graph.Draw("AL")

graph_pm.SetLineColor(ROOT.kRed)
graph_pm.Draw("LSAME")

graph_nonpm.SetLineColor(ROOT.kBlue)
graph_nonpm.Draw("LSAME")

line = ROOT.TLine(2.0, 0.0, 2.0, 1.2)
line.SetLineStyle(2)
line.Draw("SAME")

legend = ROOT.TLegend(0.5, 0.2, 0.85, 0.45)
legend.AddEntry(graph, "Purity K#rightarrow#pi^{+}#pi^{-}", "l")
legend.AddEntry(graph_pm, "Efficiency K#rightarrow#pi^{+}#pi^{-}", "l")
legend.AddEntry(graph_nonpm, "Efficiency non K#rightarrow#pi^{+}#pi^{-}", "l")
legend.SetBorderSize(0)
legend.SetFillStyle(0)
legend.Draw()
canva.Draw()

Warning in <TCanvas::Constructor>: Deleting canvas with same name: canva


### Checks of counts, purity and efficiency in the optimal threshold

In [18]:
cut_cond = "true"
optimized_variable = "Kchrec_5"
var_title = "m^{inv}_{+-} [MeV/c^{2}]"

In [19]:
basic_hist = {}
modes = {"full": [201, 280, 600], "zoom": [201, 490, 505]}

for mode, hist_config in modes.items():
    hist_list = []

    for data_type in ["PM", "nonPM", "data"]:
        h_title = f"h_{optimized_variable}_{data_type}_{mode}"
        hist_title_list = f";{var_title};Counts"

        hist_list.append(df_CS_tag[data_type].Histo1D((h_title, hist_title_list, *hist_config), optimized_variable))
    
    basic_hist[mode] = hist_list

all_ptrs = [hist for hist_list in basic_hist.values() for hist in hist_list]
ROOT.RDF.RunGraphs(all_ptrs)

# ---

# 3. Modyfikacje histogramów PO wykonaniu RunGraphs (Skalowanie i Dodawanie)
for mode, hist_list in basic_hist.items():
    # Index 0: PM, Index 1: nonPM, Index 2: data
    
    # Skalowanie histogramów MC (PM i nonPM)
    hist_list[0].Scale(scaling_factor)
    hist_list[1].Scale(scaling_factor)

    # Tworzenie sumy MC (PM + nonPM) i dodanie jej jako 4. element (indeks 3)
    h_added = hist_list[0].Clone(f"h_{optimized_variable}_added_{mode}")
    h_added.Add(hist_list[1].GetPtr(), 1.0)
    
    hist_list.append(h_added)

In [20]:
mode = "zoom"

h_calc = basic_hist[mode][0]
h_calc_obj = h_calc.GetValue() if hasattr(h_calc, "GetValue") else h_calc
mean = h_calc_obj.GetMean()
max_bin = h_calc_obj.GetMaximumBin()
max_x = h_calc_obj.GetBinCenter(max_bin)
max_y = h_calc_obj.GetBinContent(max_bin)
rms = h_calc_obj.GetRMS()
# ----------------------------------------------

# Drawing full and zoomed stacked histograms
canva = ROOT.TCanvas("canva", "", 2000, 1000)
canva.Divide(2, 1)

colors = [ROOT.kRed, ROOT.kBlue, ROOT.kBlack, ROOT.kOrange]
titles = [
    "Events with primary K#rightarrow#pi^{+}#pi^{-}", 
    "Events without primary K#rightarrow#pi^{+}#pi^{-}", 
    "Data", 
    "MC sum"
]

stacks = []
legends = []
lines = []

for pad_idx, (mode_name, hist_list) in enumerate(basic_hist.items()):
    canva.cd(pad_idx + 1)
    
    hs = ROOT.THStack(f"hs_{mode_name}", f"{mode_name.capitalize()} mode;{var_title};Counts")
    leg = ROOT.TLegend(0.45, 0.65, 0.88, 0.88)
    leg.SetBorderSize(0)
    leg.SetFillStyle(0)
    
    data_hist = None

    for h_idx, hist in enumerate(hist_list):
        h = hist.GetValue() if hasattr(hist, "GetValue") else hist
        color = colors[h_idx % len(colors)]
        title = titles[h_idx % len(titles)]
        
        # Rozpoznajemy kanał danych po tytule lub indeksie (tutaj np. indeks 2 = "Data")
        if title == "Data":
            # Styl dla punktów eksperymentalnych (PE1)
            h.SetMarkerStyle(20)       # Pełne kropki
            h.SetMarkerSize(0.8)       # Rozmiar punktów
            h.SetMarkerColor(color)
            h.SetLineColor(color)
            h.SetLineWidth(2)
            
            data_hist = h
            leg.AddEntry(h, title, "pe") # "pe" -> punkty z błędami w legendzie
        else:
            # Styl dla MC (histogramy liniowe)
            h.SetLineColor(color)
            h.SetLineWidth(2)
            
            hs.Add(h)
            leg.AddEntry(h, title, "l")  # "l" -> linia w legendzie

    line_left = ROOT.TLine(497.611 - 2.0, 0.0, 497.611 - 2.0, hs.GetMaximum() * 0.5)
    line_right = ROOT.TLine(497.611 + 2.0, 0.0, 497.611 + 2.0, hs.GetMaximum() * 0.5)

    line_right.SetLineStyle(ROOT.kDashed)
    line_left.SetLineStyle(ROOT.kDashed)
    
    # 1. Rysujemy stack z MC
    hs.Draw("nostack hist")
    line_right.Draw("SAME")
    line_left.Draw("SAME")
    
    # 2. Rysujemy Dane osobną opcją PE1 na tej samej skali
    if data_hist:
        data_hist.Draw("E1 P SAME")
        
        # Zabezpieczenie skali Y: jeśli Dane są wyższe niż MC, dopasowujemy stack
        max_y_val = max(hs.GetMaximum(), data_hist.GetMaximum()) * 0.6
        hs.SetMaximum(max_y_val)

    leg.Draw()

    stacks.append(hs)
    legends.append(leg)
    lines.append(line_right)
    lines.append(line_left)

canva._stacks = stacks
canva._legends = legends

canva.Draw()

Warning in <TCanvas::Constructor>: Deleting canvas with same name: canva


In [21]:
optimal_purity = graph.Eval(2.0)
optimal_efficiency = graph_pm.Eval(2.0)
nonpm_eff = graph_nonpm.Eval(2.0)

print(f"Optimal purity: {optimal_purity*100:.2f}%")
print(f"Optimal efficiency: {optimal_efficiency*100:.2f}%")
print(f"NonPM efficiency: {nonpm_eff*100:.2f}%")

Optimal purity: 98.00%
Optimal efficiency: 70.01%
NonPM efficiency: 1.11%


In [22]:
pm_counts_arr = np.array(pm_counts, dtype=float)
nonpm_counts_arr = np.array(nonpm_counts, dtype=float)

graph_pm_counts = ROOT.TGraph(len(x_arr), x_arr, pm_counts_arr)
graph_nonpm_counts = ROOT.TGraph(len(x_arr), x_arr, nonpm_counts_arr)

# ---

optimal_counts_pm = graph_pm_counts.Eval(2.0)
optimal_counts_nonpm = graph_nonpm_counts.Eval(2.0)

print(f"Optimal counts pm: {optimal_counts_pm:.0f}")
print(f"Optimal counts non pm: {optimal_counts_nonpm:.0f}")

Optimal counts pm: 10715219
Optimal counts non pm: 218704


In [23]:
tag_condition += " && abs(Kchrec_5 - 497.611) < 2.0"

df_CS_tag["PM"] = df_CS_tag["PM"].Filter(tag_condition)
df_CS_tag["nonPM"] = df_CS_tag["nonPM"].Filter(tag_condition)
df_CS_tag["data"] = df_CS_tag["data"].Filter(tag_condition)

total_counts_tagging = {}

total_counts_tagging["PM"] = df_CS_tag["PM"].Count()
total_counts_tagging["nonPM"] = df_CS_tag["nonPM"].Count()
total_counts_tagging["data"] = df_CS_tag["data"].Count()

ROOT.RDF.RunGraphs(list(total_counts_tagging.values()))

total_counts_tagging = {key: int(val.GetValue() * scaling_factor) if key != 'data' else int(val.GetValue()) for key, val in total_counts_tagging.items()}

In [24]:
print("Counts before tagging:")
total_MC = total_counts['PM'] + total_counts['nonPM']
data_MC_diff = (total_MC - total_counts['data']) / total_counts['data']
purity = total_counts['PM'] / (total_counts['PM'] + total_counts['nonPM'])


print(f"MC PM: {total_counts['PM']}")
print(f"MC nonPM: {total_counts['nonPM']}")
print(f"MC total: {total_MC}")
print(f"Data: {total_counts['data']}")
print(f"(MC - Data) / Data: {100 * data_MC_diff:.2f}%")
print(f"MC PM purity: {100 * purity:.2f}%")

print("\n")

print("Counts after fiducial volume:")
total_MC = total_counts_fv['PM'] + total_counts_fv['nonPM']
data_MC_diff = (total_MC - total_counts_fv['data']) / total_counts_fv['data']
eff_PM = total_counts_fv['PM']/ total_counts['PM']
eff_nonPM = total_counts_fv['nonPM']/ total_counts['nonPM']
eff_MC_total = total_MC / (total_counts['PM'] + total_counts['nonPM'])
eff_Data_total = total_counts_fv['data'] / total_counts['data']
purity = total_counts_fv['PM'] / (total_counts_fv['PM'] + total_counts_fv['nonPM'])

print(f"MC PM: {total_counts_fv['PM']} (Eff: {100 * eff_PM:.2f}%)")
print(f"MC nonPM: {total_counts_fv['nonPM']} (Eff: {100 * eff_nonPM:.2f}%)")
print(f"MC total: {total_MC} (Eff: {100 * eff_MC_total:.2f}%)")
print(f"Data: {total_counts_fv['data']} (Eff: {100 * eff_Data_total:.2f}%)")
print(f"(MC - Data) / Data: {100 * data_MC_diff:.2f}%")
print(f"MC PM purity: {100 * purity:.2f}%")

print("\n")

print("Counts after tagging:")
total_MC = total_counts_tagging['PM'] + total_counts_tagging['nonPM']
data_MC_diff = (total_MC - total_counts_tagging['data']) / total_counts_tagging['data']
eff_PM = total_counts_tagging['PM']/ total_counts['PM']
eff_nonPM = total_counts_tagging['nonPM']/ total_counts['nonPM']
eff_MC_total = total_MC / (total_counts['PM'] + total_counts['nonPM'])
eff_Data_total = total_counts_tagging['data'] / total_counts['data']
purity = total_counts_tagging['PM'] / (total_counts_tagging['PM'] + total_counts_tagging['nonPM'])

print(f"MC PM: {total_counts_tagging['PM']} (Eff: {100 * eff_PM:.2f}%)")
print(f"MC nonPM: {total_counts_tagging['nonPM']} (Eff: {100 * eff_nonPM:.2f}%)")
print(f"MC total: {total_MC} (Eff: {100 * eff_MC_total:.2f}%)")
print(f"Data: {total_counts_tagging['data']} (Eff: {100 * eff_Data_total:.2f}%)")
print(f"(MC - Data) / Data: {100 * data_MC_diff:.2f}%")
print(f"MC PM purity: {100 * purity:.2f}%")

Counts before tagging:
MC PM: 15305041
MC nonPM: 19739238
MC total: 35044279
Data: 33193796
(MC - Data) / Data: 5.57%
MC PM purity: 43.67%


Counts after fiducial volume:
MC PM: 12381856 (Eff: 80.90%)
MC nonPM: 2397526 (Eff: 12.15%)
MC total: 14779382 (Eff: 42.17%)
Data: 13746995 (Eff: 41.41%)
(MC - Data) / Data: 7.51%
MC PM purity: 83.78%


Counts after tagging:
MC PM: 10717712 (Eff: 70.03%)
MC nonPM: 218781 (Eff: 1.11%)
MC total: 10936493 (Eff: 31.21%)
Data: 9488695 (Eff: 28.59%)
(MC - Data) / Data: 15.26%
MC PM purity: 98.00%


### Create the histograms

In [25]:
cut_cond = "true"
optimized_variable = "Kchrec_5"
var_title = "m^{inv}_{+-} [MeV/c^{2}]"

In [26]:
basic_hist = {}
modes = {"full": [501, 0, 1000], "zoom": [501, 490, 505]}

for mode, hist_config in modes.items():
    hist_list = []

    for data_type in ["PM", "nonPM", "data"]:
        h_title = f"h_{optimized_variable}_{data_type}_{mode}"
        hist_title_list = f";{var_title};Counts"

        hist_list.append(df_CS_tag[data_type].Histo1D((h_title, hist_title_list, *hist_config), optimized_variable))
    
    basic_hist[mode] = hist_list

all_ptrs = [hist for hist_list in basic_hist.values() for hist in hist_list]
ROOT.RDF.RunGraphs(all_ptrs)

# ---

# 3. Modyfikacje histogramów PO wykonaniu RunGraphs (Skalowanie i Dodawanie)
for mode, hist_list in basic_hist.items():
    # Index 0: PM, Index 1: nonPM, Index 2: data
    
    # Skalowanie histogramów MC (PM i nonPM)
    hist_list[0].Scale(scaling_factor)
    hist_list[1].Scale(scaling_factor)

    # Tworzenie sumy MC (PM + nonPM) i dodanie jej jako 4. element (indeks 3)
    h_added = hist_list[0].Clone(f"h_{optimized_variable}_added_{mode}")
    h_added.Add(hist_list[1].GetPtr(), 1.0)
    
    hist_list.append(h_added)

In [27]:
mode = "zoom"

h_calc = basic_hist[mode][0]
h_calc_obj = h_calc.GetValue() if hasattr(h_calc, "GetValue") else h_calc
mean = h_calc_obj.GetMean()
max_bin = h_calc_obj.GetMaximumBin()
max_x = h_calc_obj.GetBinCenter(max_bin)
max_y = h_calc_obj.GetBinContent(max_bin)
rms = h_calc_obj.GetRMS()
# ----------------------------------------------

# Drawing full and zoomed stacked histograms
canva = ROOT.TCanvas("canva", "", 1000, 500)
canva.Divide(2, 1)

colors = [ROOT.kRed, ROOT.kBlue, ROOT.kBlack, ROOT.kOrange]
titles = [
    "Events with primary K#rightarrow#pi^{+}#pi^{-}", 
    "Events without primary K#rightarrow#pi^{+}#pi^{-}", 
    "Data", 
    "MC sum"
]

stacks = []
legends = []

for pad_idx, (mode_name, hist_list) in enumerate(basic_hist.items()):
    canva.cd(pad_idx + 1)
    
    hs = ROOT.THStack(f"hs_{mode_name}", f"{mode_name.capitalize()} mode;{var_title};Counts")
    leg = ROOT.TLegend(0.45, 0.65, 0.88, 0.88)
    leg.SetBorderSize(0)
    leg.SetFillStyle(0)
    
    data_hist = None

    for h_idx, hist in enumerate(hist_list):
        h = hist.GetValue() if hasattr(hist, "GetValue") else hist
        color = colors[h_idx % len(colors)]
        title = titles[h_idx % len(titles)]
        
        # Rozpoznajemy kanał danych po tytule lub indeksie (tutaj np. indeks 2 = "Data")
        if title == "Data":
            # Styl dla punktów eksperymentalnych (PE1)
            h.SetMarkerStyle(20)       # Pełne kropki
            h.SetMarkerSize(0.8)       # Rozmiar punktów
            h.SetMarkerColor(color)
            h.SetLineColor(color)
            h.SetLineWidth(2)
            
            data_hist = h
            leg.AddEntry(h, title, "pe") # "pe" -> punkty z błędami w legendzie
        else:
            # Styl dla MC (histogramy liniowe)
            h.SetLineColor(color)
            h.SetLineWidth(2)
            
            hs.Add(h)
            leg.AddEntry(h, title, "l")  # "l" -> linia w legendzie

    # 1. Rysujemy stack z MC
    hs.Draw("nostack hist")
    
    # 2. Rysujemy Dane osobną opcją PE1 na tej samej skali
    if data_hist:
        data_hist.Draw("E1 P SAME")
        
        # Zabezpieczenie skali Y: jeśli Dane są wyższe niż MC, dopasowujemy stack
        max_y_val = max(hs.GetMaximum(), data_hist.GetMaximum()) * 1.05
        hs.SetMaximum(max_y_val)

    leg.Draw()

    stacks.append(hs)
    legends.append(leg)

canva._stacks = stacks
canva._legends = legends

canva.Draw()

Warning in <TCanvas::Constructor>: Deleting canvas with same name: canva


## Conditional probability: KL->3pi0 if KS->pi+pi- (change of lookup to channel level)

In [28]:
df_tag_channel = {}
total_counts_tagged_channels = {}
total_counts_tagged_channels_PM = {}
total_counts_tagged_channels_nonPM = {}

for mctruth, chann in channName.items():
    if chann == 'Data':
        df_tag_channel[chann] = df_data.Filter(tag_condition)
        total_counts_tagged_channels[chann] = df_tag_channel[chann].Count()
        total_counts_tagged_channels_PM[chann] = df_tag_channel[chann].Filter("hasPM").Count()
        total_counts_tagged_channels_nonPM[chann] = df_tag_channel[chann].Filter("!hasPM").Count()
    elif chann != 'MC sum':
        df_tag_channel[chann] = df.Filter(f"mctruth == {mctruth} && ({tag_condition})")
        total_counts_tagged_channels[chann] = df_tag_channel[chann].Count()
        total_counts_tagged_channels_PM[chann] = df_tag_channel[chann].Filter("hasPM").Count()
        total_counts_tagged_channels_nonPM[chann] = df_tag_channel[chann].Filter("!hasPM").Count()

ROOT.RDF.RunGraphs(list(total_counts_tagged_channels.values()))

total_counts_tagged_channels = {key: int(val.GetValue() * scaling_factor) if key != 'Data' else int(val.GetValue()) for key, val in total_counts_tagged_channels.items()}
total_counts_tagged_channels_PM = {key: int(val.GetValue() * scaling_factor) if key != 'Data' else int(val.GetValue()) for key, val in total_counts_tagged_channels_PM.items()}
total_counts_tagged_channels_nonPM = {key: int(val.GetValue() * scaling_factor) if key != 'Data' else int(val.GetValue()) for key, val in total_counts_tagged_channels_nonPM.items()}


In [29]:
def print_purity(counts_dict, label=""):
    # Obliczamy sumę zliczeń wszystkich kanałów MC (pomijamy 'Data' oraz 'MC sum')
    total_mc = sum(
        count
        for ch, count in counts_dict.items()
        if ch not in ["Data", "MC sum"]
    )

    count_3pi0 = counts_dict.get("3pi0", 0)

    if total_mc > 0:
        purity = (count_3pi0 / total_mc) * 100
        contamination = 100.0 - purity
        print(f"--- Purity analysis {label} ---")
        print(f"3pi0 counts: {count_3pi0}")
        print(f"Total MC counts: {total_mc}")
        print(f"Purity (3pi0 / Total MC): {purity:.4f}%")
        print(f"Contamination (Background / Total MC): {contamination:.4f}%")
    else:
        print(f"--- Purity analysis {label} ---")
        print("Total MC counts is 0 - cannot calculate purity.")


# --- 1. Total counts after tagging ---
print("Total counts of channels after tagging:")
for chann in channName.values():
    print(f"Counts {chann}: {total_counts_tagged_channels.get(chann, 0)}")

print_purity(total_counts_tagged_channels, "(Total)")
print("\n" + "=" * 50 + "\n")


# --- 2. Total counts after tagging (PM) ---
print("Total counts of channels after tagging (PM):")
for chann in channName.values():
    print(f"Counts {chann}: {total_counts_tagged_channels_PM.get(chann, 0)}")

print_purity(total_counts_tagged_channels_PM, "(PM)")
print("\n" + "=" * 50 + "\n")


# --- 3. Total counts after tagging (non PM) ---
print("Total counts of channels after tagging (non PM):")
for chann in channName.values():
    print(f"Counts {chann}: {total_counts_tagged_channels_nonPM.get(chann, 0)}")

print_purity(total_counts_tagged_channels_nonPM, "(non PM)")

Total counts of channels after tagging:
Counts Data: 9488695
Counts Signal: 30066
Counts Regeneration: 76043
Counts Omega: 7819
Counts 3pi0: 8823410
Counts Semileptonic: 1544
Counts Other: 1996867
Counts pi+pi-pi+pi-: 741
Counts MC sum: 0
--- Purity analysis (Total) ---
3pi0 counts: 8823410
Total MC counts: 10936490
Purity (3pi0 / Total MC): 80.6786%
Contamination (Background / Total MC): 19.3214%


Total counts of channels after tagging (PM):
Counts Data: 0
Counts Signal: 30066
Counts Regeneration: 75921
Counts Omega: 0
Counts 3pi0: 8823410
Counts Semileptonic: 0
Counts Other: 1787572
Counts pi+pi-pi+pi-: 741
Counts MC sum: 0
--- Purity analysis (PM) ---
3pi0 counts: 8823410
Total MC counts: 10717710
Purity (3pi0 / Total MC): 82.3255%
Contamination (Background / Total MC): 17.6745%


Total counts of channels after tagging (non PM):
Counts Data: 9488695
Counts Signal: 0
Counts Regeneration: 122
Counts Omega: 7819
Counts 3pi0: 0
Counts Semileptonic: 1544
Counts Other: 209295
Counts pi+p

In [30]:
variables = ["Chi2SignalKinFit", "Knerec_5", "KnerecSix_5", "MassPi01", "MassPi02"]
var_title = ["#chi^{2}_{fit, +-00} [-]", 
             "m^{inv}_{4#gamma} [MeV/c^{2}]", 
             "m^{inv}_{6#gamma} [MeV/c^{2}]", 
             "m^{inv}_{2#gamma,1} [MeV/c^{2}]",
             "m^{inv}_{2#gamma,2} [MeV/c^{2}]"]

In [31]:
import ROOT

# Flaga sterująca: True -> odejmij tło !hasPM od Danych i bierz tylko hasPM dla MC
SUBTRACT_NON_PM = True

basic_hist = {}
modes = {
    "full": [
        [501, 0, 1000],
        [501, 0, 1000],
        [501, 2, 1000],
        [501, 0, 500],
        [501, 0, 500],
    ],
    "zoom": [
        [50, 0, 100],
        [251, 100, 600],
        [171, 300, 600],
        [251, 0, 300],
        [251, 0, 300],
    ],
}

# Słownik na pomocnicze wskaźniki histogramów !hasPM dla Danych (używany gdy SUBTRACT_NON_PM = True)
data_non_pm_ptrs = {}

for idx, (var, x_title) in enumerate(zip(variables, var_title)):
    mode_dict = {}

    for mode, pars in modes.items():

        hist_dict = {}
        for chann in channName.values():
            if chann == "MC sum":
                continue

            h_title = f"hist_{var}_{mode}_{chann}"
            hist_title_list = f";{x_title};Counts"

            # Przypisujemy pomocniczy dataframe dla danego kanału
            df_aux = df_tag_channel[chann]

            if chann == "Data":
                # Dla Danych tworzymy standardowy histogram
                hist_dict[chann] = df_aux.Histo1D(
                    (h_title, hist_title_list, *pars[idx]), var
                )

                # Jeśli odejmujemy !hasPM, tworzymy dodatkowy histogram filtracyjny dla Data
                if SUBTRACT_NON_PM:
                    h_title_non_pm = f"hist_{var}_{mode}_{chann}_non_pm"
                    df_non_pm_aux = df_aux.Filter("!hasPM")
                    data_non_pm_ptrs[(var, mode)] = df_non_pm_aux.Histo1D(
                        (h_title_non_pm, hist_title_list, *pars[idx]), var
                    )
            else:
                # Dla MC: jeśli włączone odejmowanie, bierzemy tylko zdarzenia z hasPM
                if SUBTRACT_NON_PM:
                    df_aux = df_aux.Filter("hasPM")

                hist_dict[chann] = df_aux.Histo1D(
                    (h_title, hist_title_list, *pars[idx]), var
                )

        mode_dict[mode] = hist_dict

    basic_hist[var] = mode_dict

# 2. Ewaluacja wszystkich grafów na raz (Lazy Evaluation)
all_ptrs = [
    h_ptr
    for mode_d in basic_hist.values()
    for ch_d in mode_d.values()
    for h_ptr in ch_d.values()
]
if SUBTRACT_NON_PM:
    all_ptrs.extend(data_non_pm_ptrs.values())

ROOT.RDF.RunGraphs(all_ptrs)

# 3. Odejmowanie tła !hasPM od Danych, skalowanie MC i tworzenie "MC sum"
for var, mode_d in basic_hist.items():
    for mode, ch_d in mode_d.items():

        # Odejmujemy histogram !hasPM od histogramu Danych
        if SUBTRACT_NON_PM and "Data" in ch_d:
            h_data = ch_d["Data"].GetPtr()
            h_data_non_pm = data_non_pm_ptrs[(var, mode)].GetPtr()
            h_data.Add(h_data_non_pm, -1.0)

        # Skalowanie kanałów MC (wszystko poza 'Data')
        for chann, h_ptr in ch_d.items():
            if chann != "Data":
                h_ptr.Scale(scaling_factor)

        # Tworzenie sumy MC
        mc_channels = [ch for ch in ch_d.keys() if ch != "Data"]

        if mc_channels:
            h_mc_sum = ch_d[mc_channels[0]].Clone(f"hist_{var}_{mode}_MC_sum")
            for ch in mc_channels[1:]:
                h_mc_sum.Add(ch_d[ch].GetPtr(), 1.0)

            # Zapisujemy wygenerowaną sumę w słowniku
            ch_d["MC sum"] = h_mc_sum

In [32]:
# Określamy wymiary siatki
n_rows = len(variables)
n_cols = len(modes)

# Tworzymy płótno przeskalowane do liczby wierszy i kolumn
canva = ROOT.TCanvas("canva", "Histograms", 500 * n_cols, 400 * n_rows)
canva.Divide(n_cols, n_rows)

stacks = []
legends = []

# Pętla po wierszach (zmienne) i kolumnach (tryby)
for row_idx, (var, x_title) in enumerate(zip(variables, var_title)):
    for col_idx, (mode_name, chann_dict) in enumerate(basic_hist[var].items()):
        
        # Obliczenie indeksu podokna w TCanvas (ROOT numeruje od 1: wiersz po wierszu)
        pad_idx = row_idx * n_cols + col_idx + 1
        canva.cd(pad_idx)
        
        hs = ROOT.THStack(f"hs_{var}_{mode_name}", f"{mode_name.capitalize()} mode;{x_title};Counts")
        leg = ROOT.TLegend(0.45, 0.65, 0.88, 0.88)
        leg.SetBorderSize(0)
        leg.SetFillStyle(0)
        
        data_hist = None

        for chann_key, hist in chann_dict.items():
            h = hist.GetValue() if hasattr(hist, "GetValue") else hist
            color = channColor.get(chann_key, ROOT.kBlack)
            title = channTitle.get(chann_key, chann_key)
            
            if chann_key == "Data":
                # Styl dla danych eksperymentalnych (PE1)
                h.SetMarkerStyle(20)
                h.SetMarkerSize(0.8)
                h.SetMarkerColor(color)
                h.SetLineColor(color)
                h.SetLineWidth(2)
                
                data_hist = h
                leg.AddEntry(h, title, "pe")
            else:
                # Styl dla MC
                h.SetLineColor(color)
                h.SetLineWidth(2)
                
                hs.Add(h)
                leg.AddEntry(h, title, "l")

        # 1. Rysujemy stack z MC
        hs.Draw("nostack hist")
        
        # 2. Rysujemy Dane oraz dopasowujemy skalę Y
        if data_hist:
            data_hist.Draw("E1 P SAME")
            max_y_val = max(hs.GetMaximum(), data_hist.GetMaximum()) * 0.6
            hs.SetMaximum(max_y_val)

        leg.Draw()

        stacks.append(hs)
        legends.append(leg)

# Zapobieganie usunięciu obiektów z pamięci przez Python Garbage Collector
canva._stacks = stacks
canva._legends = legends

canva.Draw()

Warning in <TCanvas::Constructor>: Deleting canvas with same name: canva


## Application of cuts

In [45]:
mctruth_signal = 4

df_tag_3pi0 = df.Filter(f"mctruth == {mctruth_signal} && {tag_condition} && hasPM")
df_tag_non_3pi0 = df.Filter(f"mctruth != {mctruth_signal} && {tag_condition} && hasPM")
df_tag_data = df_data.Filter(f"{tag_condition}")

In [49]:
import math

pi0Mass1_mean = 134.83240924168848
pi0Mass2_mean = 134.87134080668446

expr_u = f"((MassPi01 - {pi0Mass1_mean}) + (MassPi02 - {pi0Mass2_mean})) / sqrt(2)"
expr_v = f"((MassPi01 - {pi0Mass1_mean}) - (MassPi02 - {pi0Mass2_mean})) / sqrt(2)"


def apply_uv_definitions(df):
    cols = [str(c) for c in df.GetColumnNames()]

    # Wybór odpowiedniej metody w zależności od tego, czy kolumna już istnieje
    def_u = df.Redefine if "u" in cols else df.Define
    df = def_u("u", expr_u)

    cols = [str(c) for c in df.GetColumnNames()]
    def_v = df.Redefine if "v" in cols else df.Define
    df = def_v("v", expr_v)

    return df


# Aplikacja dla słownika
for chann, df_part in df_tag_channel.items():
    df_tag_channel[chann] = apply_uv_definitions(df_part)

# Aplikacja dla pozostałych RDataFrame
df_tag_3pi0 = apply_uv_definitions(df_tag_3pi0)
df_tag_non_3pi0 = apply_uv_definitions(df_tag_non_3pi0)
df_tag_data = apply_uv_definitions(df_tag_data)

In [35]:
ROOT.gInterpreter.Declare("""
bool ellipse_cut(double u, double v) {
    double u0 = -92.9524, v0 = 7.1644, su = 33.7633, sv = 42.0163, rho = 0.556;
    double n_sigma_cut = 2.5;
    
    double du = u - u0;
    double dv = v - v0;
    double inv_rho2 = 1.0 / (1.0 - rho * rho);
    double dist2 = inv_rho2 * ((du * du) / (su * su) + (dv * dv) / (sv * sv) - 2.0 * rho * (du * dv) / (su * sv));
    return std::sqrt(std::max(0.0, dist2)) > n_sigma_cut;
}
""")

True

In [47]:
# 2. Add your cuts, including any that use u and v
cuts = [
    "Chi2SignalKinFit < 30.0",
    "ellipse_cut(Knerec_5 - 497.611, KnerecSix_5 - 497.611)",
    "abs(u) < 8.085754304 && abs(v) < 11.514665004",
    "(rho_fid_omega < 2.05 && abs(z_fid_omega) < 2.45 && sqrt(rho_ch*rho_ch + rho_ne*rho_ne) > 0.8) || !(rho_fid_omega < 2.05 && abs(z_fid_omega) < 2.45)"
]

cut_counts = []
cut_counts_bkg = []
cut_eff = []
rel_cut_eff = []

basis_3pi0_counts = total_counts_tagged_channels["3pi0"]

purity = []

combined_cut = ""

for cut in cuts:
    combined_cut += f" && ({cut})" if len(combined_cut) > 0 else f"({cut})"
    cut_counts.append(df_tag_3pi0.Filter(combined_cut).Count())
    cut_counts_bkg.append(df_tag_non_3pi0.Filter(combined_cut).Count())

tot_cuts = cut_counts + cut_counts_bkg
ROOT.RDF.RunGraphs(tot_cuts)

for i, count in enumerate(cut_counts):
    cut_counts[i] = count.GetValue()
    cut_counts_bkg[i] = cut_counts_bkg[i].GetValue()
    eff = cut_counts[i] / basis_3pi0_counts if basis_3pi0_counts > 0 else 1.0
    rel_eff = cut_counts[i] / cut_counts[i-1] if i > 0 and cut_counts[i-1] > 0 else eff
    pur = (cut_counts[i]) / (cut_counts[i] + cut_counts_bkg[i]) if (cut_counts[i] + cut_counts_bkg[i]) != 0 else 1.0
    cut_eff.append(eff)
    rel_cut_eff.append(rel_eff)
    purity.append(pur)

    print(f"Cut {cuts[i]}: Counts {int(cut_counts[i]*scaling_factor)}, Efficiency {cut_eff[i]*100:.3f}%, Relative efficiency {rel_cut_eff[i]*100:.3f}%, Counts {int(cut_counts_bkg[i]*scaling_factor)}, Purity {purity[i]*100:.3f}%")



Cut Chi2SignalKinFit < 30.0: Counts 690054, Efficiency 11.468%, Relative efficiency 11.468%, Counts 26903, Purity 96.248%
Cut ellipse_cut(Knerec_5 - 497.611, KnerecSix_5 - 497.611): Counts 260797, Efficiency 4.334%, Relative efficiency 37.794%, Counts 26060, Purity 90.915%
Cut abs(u) < 8.085754304 && abs(v) < 11.514665004: Counts 2427, Efficiency 0.040%, Relative efficiency 0.931%, Counts 14643, Purity 14.222%
Cut (rho_fid_omega < 2.05 && abs(z_fid_omega) < 2.45 && sqrt(rho_ch*rho_ch + rho_ne*rho_ne) > 0.8) || !(rho_fid_omega < 2.05 && abs(z_fid_omega) < 2.45): Counts 2419, Efficiency 0.040%, Relative efficiency 99.663%, Counts 14523, Purity 14.281%


In [50]:
# # 2. Add your cuts, including any that use u and v

cut_counts = []
cut_eff = []
rel_cut_eff = []

combined_cut = ""

basis_data_counts = total_counts_tagged_channels["Data"] - 


for cut in cuts:
    combined_cut += f" && ({cut})" if len(combined_cut) > 0 else f"({cut})"
    cut_counts.append(df_tag_data.Filter(combined_cut).Count())

ROOT.RDF.RunGraphs(cut_counts)

for i, count in enumerate(cut_counts):
    cut_counts[i] = count.GetValue()
    eff = cut_counts[i] / basis_data_counts if basis_data_counts > 0 else 1.0
    rel_eff = cut_counts[i] / cut_counts[i-1] if i > 0 and cut_counts[i-1] > 0 else eff
    cut_eff.append(eff)
    rel_cut_eff.append(rel_eff)

    print(f"Cut {cuts[i]}: Counts {cut_counts[i]}, Efficiency {cut_eff[i]*100:.3f}%, Relative efficiency {rel_cut_eff[i]*100:.3f}%")

Cut Chi2SignalKinFit < 30.0: Counts 663582, Efficiency 6.993%, Relative efficiency 6.993%
Cut ellipse_cut(Knerec_5 - 497.611, KnerecSix_5 - 497.611): Counts 260958, Efficiency 2.750%, Relative efficiency 39.326%
Cut abs(u) < 8.085754304 && abs(v) < 11.514665004: Counts 14554, Efficiency 0.153%, Relative efficiency 5.577%
Cut (rho_fid_omega < 2.05 && abs(z_fid_omega) < 2.45 && sqrt(rho_ch*rho_ch + rho_ne*rho_ne) > 0.8) || !(rho_fid_omega < 2.05 && abs(z_fid_omega) < 2.45): Counts 14312, Efficiency 0.151%, Relative efficiency 98.337%


In [38]:
variables = ["Chi2SignalKinFit", "Knerec_5", "KnerecSix_5", "MassPi01", "MassPi02", "Kchrec_5"]
var_title = ["#chi^{2}_{fit, +-00} [-]", 
             "m^{inv}_{4#gamma} [MeV/c^{2}]", 
             "m^{inv}_{6#gamma} [MeV/c^{2}]", 
             "m^{inv}_{2#gamma,1} [MeV/c^{2}]",
             "m^{inv}_{2#gamma,2} [MeV/c^{2}]",
             "m^{inv}_{+-} [MeV/c^{2}]"]

In [39]:
import ROOT

# Flaga sterująca: True -> odejmij tło MC z !hasPM od Danych i zachowaj tylko MC z hasPM
SUBTRACT_NON_PM = True

combined_cut = """true"""

# '''Chi2SignalKinFit < 30. &&
#                  ellipse_cut(Knerec_5 - 497.611, KnerecSix_5 - 497.611) &&
#                  (abs(u) < 8.085754304 && abs(v) < 11.514665004) &&
#                  ((rho_fid_omega < 2.05 && abs(z_fid_omega) < 2.45 && sqrt(rho_ch*rho_ch + rho_ne*rho_ne) > 0.8) || !(rho_fid_omega < 2.05 && abs(z_fid_omega) < 2.45))
#               '''

basic_hist = {}
modes = {
    "full": [
        [501, 0.0, 1000],
        [501, 0, 1000],
        [501, 2, 1000],
        [501, 0, 500],
        [501, 0, 500],
        [201, 496, 499],
    ],
    "zoom": [
        [101, 0.0, 100],
        [251, 0, 600],
        [171, 300, 600],
        [251, 0, 300],
        [251, 0, 300],
        [201, 496, 499],
    ],
}

# Słownik na pomocnicze wskaźniki histogramów MC !hasPM (do odjęcia od Danych)
mc_non_pm_ptrs = {}

for idx, (var, x_title) in enumerate(zip(variables, var_title)):
    mode_dict = {}

    for mode, pars in modes.items():

        hist_dict = {}
        non_pm_list = []

        for chann in channName.values():
            if chann == "MC sum":
                continue

            h_title = f"hist_{var}_{mode}_{chann}"
            hist_title_list = f";{x_title};Counts"

            # Pomocnicza zmienna RDataFrame
            df_aux = df_tag_channel[chann].Filter(combined_cut)

            if chann == "Data":
                # Dla Danych po prostu pobieramy histogram
                hist_dict[chann] = df_aux.Histo1D(
                    (h_title, hist_title_list, *pars[idx]), var
                )
            else:
                if SUBTRACT_NON_PM:
                    # Tło MC z !hasPM rejestrujemy osobno do odjęcia od Danych
                    h_title_non_pm = f"hist_{var}_{mode}_{chann}_non_pm"
                    df_non_pm_aux = df_aux.Filter("!hasPM")
                    non_pm_list.append(
                        df_non_pm_aux.Histo1D(
                            (h_title_non_pm, hist_title_list, *pars[idx]), var
                        )
                    )

                    # Do głównego słownika trafia tylko MC z hasPM
                    df_aux = df_aux.Filter("hasPM")

                hist_dict[chann] = df_aux.Histo1D(
                    (h_title, hist_title_list, *pars[idx]), var
                )

        if SUBTRACT_NON_PM:
            mc_non_pm_ptrs[(var, mode)] = non_pm_list

        mode_dict[mode] = hist_dict

    basic_hist[var] = mode_dict

# 2. Ewaluacja wszystkich grafów na raz (Lazy Evaluation)
all_ptrs = [
    h_ptr
    for mode_d in basic_hist.values()
    for ch_d in mode_d.values()
    for h_ptr in ch_d.values()
]
if SUBTRACT_NON_PM:
    all_ptrs.extend(
        [h_ptr for ptr_list in mc_non_pm_ptrs.values() for h_ptr in ptr_list]
    )

ROOT.RDF.RunGraphs(all_ptrs)

# 3. Odejmowanie tła MC (!hasPM) od Danych, skalowanie MC i tworzenie "MC sum"
for var, mode_d in basic_hist.items():
    for mode, ch_d in mode_d.items():

        # Odejmujemy przeskalowane tło MC (!hasPM) od histogramu Danych
        if SUBTRACT_NON_PM and "Data" in ch_d:
            h_data = ch_d["Data"].GetPtr()
            for h_non_pm_ptr in mc_non_pm_ptrs[(var, mode)]:
                h_data.Add(
                    h_non_pm_ptr.GetPtr(), -1.0 * scaling_factor
                )  # Odjęcie z uwzględnieniem skalowania MC

        # Skalowanie głównych kanałów MC (wszystko poza 'Data')
        for chann, h_ptr in ch_d.items():
            if chann != "Data":
                h_ptr.Scale(scaling_factor)

        # Tworzenie sumy MC
        mc_channels = [ch for ch in ch_d.keys() if ch != "Data"]

        if mc_channels:
            h_mc_sum = ch_d[mc_channels[0]].Clone(f"hist_{var}_{mode}_MC_sum")
            for ch in mc_channels[1:]:
                h_mc_sum.Add(ch_d[ch].GetPtr(), 1.0)

            # Zapisujemy wygenerowaną sumę w słowniku
            ch_d["MC sum"] = h_mc_sum

In [40]:
canvases = []
stacks = []
legends = []

# Pętla po zmiennych – każda zmienna tworzy nowy Canvas
for row_idx, (var, x_title) in enumerate(zip(variables, var_title)):

    # Tworzenie osobnego TCanvas podzielonego na kolumny dla poszczególnych trybów (mode)
    canva = ROOT.TCanvas(f"canva_{var}", f"Canvas {var}", 500 * n_cols, 400)
    canva.Divide(n_cols, 1)

    for col_idx, (mode_name, chann_dict) in enumerate(basic_hist[var].items()):

        canva.cd(col_idx + 1)

        hs = ROOT.THStack(
            f"hs_{var}_{mode_name}",
            f"{mode_name.capitalize()} mode;{x_title};Counts",
        )
        leg = ROOT.TLegend(0.45, 0.65, 0.88, 0.88)
        leg.SetBorderSize(0)
        leg.SetFillStyle(0)

        data_hist = None

        for chann_key, hist in chann_dict.items():
            h = hist.GetValue() if hasattr(hist, "GetValue") else hist
            color = channColor.get(chann_key, ROOT.kBlack)
            title = channTitle.get(chann_key, chann_key)

            if chann_key == "Data":
                # Styl dla danych eksperymentalnych (PE1)
                h.SetMarkerStyle(20)
                h.SetMarkerSize(0.8)
                h.SetMarkerColor(color)
                h.SetLineColor(color)
                h.SetLineWidth(2)

                data_hist = h
                leg.AddEntry(h, title, "pe")
            else:
                # Styl dla MC
                h.SetLineColor(color)
                h.SetLineWidth(2)

                hs.Add(h)
                leg.AddEntry(h, title, "l")

        # 1. Rysujemy stack z MC
        hs.Draw("nostack hist")

        # 2. Rysujemy Dane oraz dopasowujemy skalę Y
        if data_hist:
            data_hist.Draw("E1 P SAME")
            max_y_val = max(hs.GetMaximum(), data_hist.GetMaximum()) * 0.6
            hs.SetMaximum(max_y_val)

        leg.Draw()

        stacks.append(hs)
        legends.append(leg)

    canva.Draw()
    canvases.append(canva)

# Zapobieganie usunięciu obiektów z pamięci przez Python Garbage Collector
for c in canvases:
    c._stacks = stacks
    c._legends = legends

In [41]:
import ctypes
import ROOT

canvases = []

ROOT.Math.MinimizerOptions.SetDefaultMinimizer("Minuit", "Migrad")

for row_idx, (var, x_title) in enumerate(zip(variables, var_title)):

    canva = ROOT.TCanvas(f"canva_{var}", f"Canvas {var}", 500 * n_cols, 600)
    canva.Divide(n_cols, 1)

    canva_stacks = []
    canva_legends = []
    canva_fitted = []
    canva_pads = []

    # Wyciągamy dane z trybu Full na początku, aby wyliczyć współczynniki fitu
    full_chann_dict = basic_hist[var].get(
        "full", next(iter(basic_hist[var].values()))
    )

    data_hist_full = None
    sig_hist_full = None
    bkg_hist_full = None
    bkg_list_full = []

    for chann_key, hist in full_chann_dict.items():
        h = hist.GetValue() if hasattr(hist, "GetValue") else hist
        if chann_key in ["Data", "data"]:
            data_hist_full = h
        elif chann_key in ["PM", "3pi0"]:
            sig_hist_full = h
        elif chann_key != "MC sum":
            bkg_list_full.append(h)

    if bkg_list_full:
        bkg_hist_full = bkg_list_full[0].Clone(f"bkg_raw_full_{var}")
        for h_bkg in bkg_list_full[1:]:
            bkg_hist_full.Add(h_bkg)

    # ----------------------------------------------------
    # FIT TFractionFitter TYLKO NA FULL MODE
    # ----------------------------------------------------
    frac_sig, err_sig = 0.0, 0.0
    frac_bkg, err_bkg = 0.0, 0.0

    if data_hist_full and sig_hist_full and bkg_hist_full:
        h_sig_fit = sig_hist_full.Clone(f"{sig_hist_full.GetName()}_fit")
        h_bkg_fit = bkg_hist_full.Clone(f"{bkg_hist_full.GetName()}_fit")
        h_data_fit = data_hist_full.Clone(f"{data_hist_full.GetName()}_fit")

        for h in [h_sig_fit, h_bkg_fit, h_data_fit]:
            for b in range(1, h.GetNbinsX() + 1):
                if h.GetBinContent(b) < 0:
                    h.SetBinContent(b, 0)
                    h.SetBinError(b, 0)

        mc_array = ROOT.TObjArray(2)
        mc_array.Add(h_sig_fit)
        mc_array.Add(h_bkg_fit)

        fitter = ROOT.TFractionFitter(h_data_fit, mc_array)
        status = fitter.Fit()

        print(f"\n=== TFractionFitter Result (Full mode fit): {var} ===")
        if int(status) == 0:
            val_s, err_s = ctypes.c_double(0), ctypes.c_double(0)
            val_b, err_b = ctypes.c_double(0), ctypes.c_double(0)

            fitter.GetResult(0, val_s, err_s)
            fitter.GetResult(1, val_b, err_b)

            frac_sig, err_sig = val_s.value, err_s.value
            frac_bkg, err_bkg = val_b.value, err_b.value

            print(f"Signal (mctruth=4) : {frac_sig:.4f} +/- {err_sig:.4f}")
            print(f"Bkg    (mctruth!=4): {frac_bkg:.4f} +/- {err_bkg:.4f}")
        else:
            print(f"Brak zbieżności fitu na Full mode! Status = {status}")

    # ----------------------------------------------------
    # RYSOWANIE PANELI (Full oraz Zoom na wyznaczonych frakcjach)
    # ----------------------------------------------------
    for col_idx, (mode_name, chann_dict) in enumerate(basic_hist[var].items()):

        canva.cd(col_idx + 1)

        pad1 = ROOT.TPad(
            f"pad1_{var}_{mode_name}",
            f"pad1_{var}_{mode_name}",
            0.0,
            0.3,
            1.0,
            1.0,
        )
        pad2 = ROOT.TPad(
            f"pad2_{var}_{mode_name}",
            f"pad2_{var}_{mode_name}",
            0.0,
            0.0,
            1.0,
            0.3,
        )

        pad1.SetBottomMargin(0.02)
        pad2.SetTopMargin(0.02)
        pad2.SetBottomMargin(0.35)

        pad1.Draw()
        pad2.Draw()

        canva_pads.extend([pad1, pad2])

        pad1.cd()

        hs = ROOT.THStack(
            f"hs_{var}_{mode_name}",
            f"{mode_name.capitalize()} mode;;Counts",
        )
        leg = ROOT.TLegend(0.40, 0.55, 0.88, 0.88)
        leg.SetBorderSize(0)
        leg.SetFillStyle(0)

        data_hist = None
        sig_hist_raw = None
        bkg_hists_raw = []

        for chann_key, hist in chann_dict.items():
            h = hist.GetValue() if hasattr(hist, "GetValue") else hist
            if chann_key in ["Data", "data"]:
                data_hist = h
            elif chann_key in ["PM", "3pi0"]:
                sig_hist_raw = h
            elif chann_key != "MC sum":
                bkg_hists_raw.append((chann_key, h))

        bkg_hist_raw = None
        if bkg_hists_raw:
            bkg_hist_raw = bkg_hists_raw[0][1].Clone(
                f"bkg_raw_{var}_{mode_name}"
            )
            for _, h_bkg in bkg_hists_raw[1:]:
                bkg_hist_raw.Add(h_bkg)

        # Skalowanie na podstawie frakcji obliczonych w Full mode
        if data_hist:
            n_data = data_hist.Integral()
            if sig_hist_raw and sig_hist_raw.Integral() > 0:
                sig_hist_raw.Scale(
                    (frac_sig * n_data) / sig_hist_raw.Integral()
                )
            if bkg_hist_raw and bkg_hist_raw.Integral() > 0:
                bkg_hist_raw.Scale(
                    (frac_bkg * n_data) / bkg_hist_raw.Integral()
                )

        # Tworzenie sumy fitu dla aktualnego widoku
        h_fit_sum = None
        if sig_hist_raw and bkg_hist_raw:
            h_fit_sum = sig_hist_raw.Clone(f"h_fit_sum_{var}_{mode_name}")
            h_fit_sum.Add(bkg_hist_raw)
            h_fit_sum.SetDirectory(0)
            canva_fitted.append(h_fit_sum)

        if sig_hist_raw:
            color = channColor.get("PM", ROOT.kRed)
            sig_hist_raw.SetLineColor(color)
            sig_hist_raw.SetLineWidth(2)
            hs.Add(sig_hist_raw)
            channel_name = "K_{S}K_{L}#rightarrow#pi^{+}#pi^{-}3#pi^{0}"
            leg.AddEntry(
                sig_hist_raw,
                f"{channel_name} ({frac_sig*100:.1f} #pm {err_sig*100:.1f}%)",
                "l",
            )

        if bkg_hist_raw:
            color = channColor.get("nonPM", ROOT.kBlue)
            bkg_hist_raw.SetLineColor(color)
            bkg_hist_raw.SetLineWidth(2)
            hs.Add(bkg_hist_raw)
            leg.AddEntry(
                bkg_hist_raw,
                f"Other channels ({frac_bkg*100:.1f} #pm {err_bkg*100:.1f}%)",
                "l",
            )

        if data_hist:
            color = channColor.get("Data", ROOT.kBlack)
            data_hist.SetMarkerStyle(20)
            data_hist.SetMarkerSize(0.8)
            data_hist.SetMarkerColor(color)
            data_hist.SetLineColor(color)
            data_hist.SetLineWidth(2)
            leg.AddEntry(data_hist, "Data", "pe")

        hs.Draw("nostack hist")

        if h_fit_sum:
            h_fit_sum.SetLineColor(ROOT.kOrange + 1)
            h_fit_sum.SetLineWidth(2)
            h_fit_sum.Draw("HIST SAME")
            leg.AddEntry(h_fit_sum, "Fit sum", "l")

        if data_hist:
            data_hist.Draw("E1 P SAME")
            max_y_val = max(hs.GetMaximum(), data_hist.GetMaximum()) * 1.25
            hs.SetMaximum(max_y_val)

        hs.GetXaxis().SetLabelSize(0)
        hs.GetXaxis().SetTitleSize(0)

        # Dolny panel (Pad 2) - Residua oraz wyliczanie Chi2
        pad2.cd()
        if h_fit_sum and data_hist:
            h_res = h_fit_sum.Clone(f"h_res_{var}_{mode_name}")
            h_res.SetDirectory(0)

            chi2 = 0.0
            n_bins_used = 0

            for b in range(1, h_res.GetNbinsX() + 1):
                fit_val = h_fit_sum.GetBinContent(b)
                data_val = data_hist.GetBinContent(b)
                data_err = data_hist.GetBinError(b)

                if data_err > 0:
                    diff_norm = (fit_val - data_val) / data_err
                    h_res.SetBinContent(b, diff_norm)
                    h_res.SetBinError(b, 1.0)

                    chi2 += diff_norm**2
                    n_bins_used += 1
                else:
                    h_res.SetBinContent(b, 0)
                    h_res.SetBinError(b, 0)

            n_params = 2  # Wyznaczone frakcje: signal + background
            ndf = max(1, n_bins_used - n_params)
            chi2_ndf = chi2 / ndf

            print(
                f"[{var} - {mode_name}] Chi2 = {chi2:.2f}, ndf = {ndf}, Chi2/ndf = {chi2_ndf:.2f}"
            )

            # Dodanie informacji o Chi2 do legendy na pierwszym padzie
            pad1.cd()
            leg.AddEntry(
                "", f"#chi^{{2}}/ndf = {chi2:.1f}/{ndf} = {chi2_ndf:.2f}", ""
            )
            pad2.cd()

            h_res.SetTitle(f";{x_title};(Fit - Data) / #sigma")
            h_res.SetLineColor(ROOT.kBlack)
            h_res.SetLineWidth(1)
            h_res.SetMarkerStyle(20)
            h_res.SetMarkerSize(0.6)

            h_res.GetXaxis().SetTitleSize(0.12)
            h_res.GetXaxis().SetLabelSize(0.10)
            h_res.GetYaxis().SetTitleSize(0.09)
            h_res.GetYaxis().SetLabelSize(0.08)
            h_res.GetYaxis().SetTitleOffset(0.55)
            h_res.GetYaxis().SetNdivisions(505)

            h_res.Draw("E1 P")

            line = ROOT.TLine(
                h_res.GetXaxis().GetXmin(), 0, h_res.GetXaxis().GetXmax(), 0
            )
            line.SetLineColor(ROOT.kRed)
            line.SetLineStyle(2)
            line.SetLineWidth(2)
            line.Draw("SAME")

            canva_fitted.extend([h_res, line])

        pad1.cd()
        leg.Draw()

        canva_stacks.append(hs)
        canva_legends.append(leg)

    canva._stacks = canva_stacks
    canva._legends = canva_legends
    canva._fitted = canva_fitted
    canva._pads = canva_pads

    canva.Update()
    out_filename = f"fit_result_{var}.png"
    canva.SaveAs(out_filename)
    print(f"-> Zapisano wykres: {out_filename}")

    canva.Draw()
    canvases.append(canva)


=== TFractionFitter Result (Full mode fit): Chi2SignalKinFit ===
Signal (mctruth=4) : 0.9830 +/- 0.0008
Bkg    (mctruth!=4): 0.0170 +/- 0.0004
[Chi2SignalKinFit - full] Chi2 = 1115.57, ndf = 499, Chi2/ndf = 2.24
[Chi2SignalKinFit - zoom] Chi2 = 422.21, ndf = 99, Chi2/ndf = 4.26
-> Zapisano wykres: fit_result_Chi2SignalKinFit.png

=== TFractionFitter Result (Full mode fit): Knerec_5 ===
Signal (mctruth=4) : 0.8997 +/- 0.0005
Bkg    (mctruth!=4): 0.1003 +/- 0.0003
[Knerec_5 - full] Chi2 = 13873.78, ndf = 499, Chi2/ndf = 27.80
[Knerec_5 - zoom] Chi2 = 11660.31, ndf = 249, Chi2/ndf = 46.83
-> Zapisano wykres: fit_result_Knerec_5.png

=== TFractionFitter Result (Full mode fit): KnerecSix_5 ===
Signal (mctruth=4) : 0.9753 +/- 0.0006
Bkg    (mctruth!=4): 0.0247 +/- 0.0003
[KnerecSix_5 - full] Chi2 = 26729.07, ndf = 349, Chi2/ndf = 76.59
[KnerecSix_5 - zoom] Chi2 = 26948.00, ndf = 169, Chi2/ndf = 159.46
-> Zapisano wykres: fit_result_KnerecSix_5.png

=== TFractionFitter Result (Full mode fit)

Warning in <TCanvas::Constructor>: Deleting canvas with same name: canva_Chi2SignalKinFit
Info in <TCanvas::Print>: png file fit_result_Chi2SignalKinFit.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canva_Knerec_5
Info in <TCanvas::Print>: png file fit_result_Knerec_5.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canva_KnerecSix_5
Info in <TCanvas::Print>: png file fit_result_KnerecSix_5.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canva_MassPi01
Info in <TCanvas::Print>: png file fit_result_MassPi01.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canva_MassPi02
Info in <TCanvas::Print>: png file fit_result_MassPi02.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canva_Kchrec_5
Warning in <TFractionFitter::Fit>: Abnormal termination of minimization.
Info in <TCanvas::Print>: png fil